# Python System Interpreter

This notebook introduces a common Python packaging problem on a shared Linux machine. Bob and Alice both work on the same computer and both rely on the same system interpreter. Bob is the administrator for this shared machine, so he can install or replace packages in the system Python environment. Alice is a normal user, so she cannot modify system packages directly. This difference matters because the Python interpreter is shared, but the users and their home directories are separate.

| User    | Home           | Sudo |
| ------- | -------------- | ---- |
| `bob`   | `/home/bob`    | yes  |
| `alice` | `/home/alice`  | no   |

This arrangement creates dependency conflicts quickly. A package change made for one user can break the other user's project, and Alice's own work becomes hard to manage when her multiple projects need different versions of the same dependency. One shared Python environment cannot safely satisfy all of those conflicting requirements at the same time.

This notebook focuses on these **dependency-conflict scenarios**:

- **Shared machine conflict.** Two users share one system Python installation.
- **Cross-user conflict.** Bob's legacy dependency can be replaced by a package change made for Alice.
- **Cross-project conflict.** Alice's own projects can still fight over incompatible versions when they share one user-level environment.
- **Isolation progression.** We then compare system packages, `pip install --user`, and project-specific virtual environments.

---

## Helper Functions

First, we define three small helper functions that keep the notebook examples compact. Each command helper prints the exact Bash command it executes, so you can compare the Python call with the equivalent command you would type in a Bash terminal.

- `run_bash_cmd(command)`: runs a Bash command as the current notebook user. Example: `run_bash_cmd("echo 'hello'")` is equivalent to running `echo 'hello'` in a Bash terminal.
- `run_bash_cmd_as_user(user, command)`: runs a Bash command as another Linux user. Example: `run_bash_cmd_as_user("alice", "whoami")` is equivalent to `sudo -H -u alice bash -lc 'whoami'`.
- `write_project(root, owner, code)`: writes one embedded demo application to disk, places it in the right project directory, and assigns ownership to the intended Linux user.


In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys


def run_bash_cmd(command: str, *, title: str = '') -> subprocess.CompletedProcess[str]:
    """Run a Bash command and print the command, output, and exit code.

    Nonzero exit codes stay visible, so command failures remain part of the
    lesson instead of being hidden.
    """

    if title:
        print(title)
        print('-' * len(title))
    print(f'$ {command}')
    completed = subprocess.run(
        ['bash', '-lc', command],
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout.rstrip())
    if completed.stderr:
        print('[stderr]')
        print(completed.stderr.rstrip())
    print(f'[exit {completed.returncode}]\n')
    return completed


def run_bash_cmd_as_user(user: str, command: str, *, title: str = '') -> subprocess.CompletedProcess[str]:
    """Run a Bash command as another Linux user through passwordless sudo.

    This helper builds the equivalent `sudo -H -u ... bash -lc ...` command
    and then delegates the printing and execution details to `run_bash_cmd`.
    """

    user_command = f"sudo -H -u {shlex.quote(user)} bash -lc {shlex.quote(command)}"
    return run_bash_cmd(user_command, title=title or f'As {user}')


def write_project(root: Path, owner: str, code: str) -> None:
    """Write an embedded demo app into a user-owned project directory."""

    root_arg = shlex.quote(str(root))
    parent_arg = shlex.quote(str(root.parent))
    tmp = Path('/tmp') / f'{root.name}-main.py'
    tmp.write_text(code)

    run_bash_cmd(f'sudo mkdir -p {root_arg}', title=f'mkdir -p {root}')
    run_bash_cmd(f'sudo cp {shlex.quote(str(tmp))} {shlex.quote(str(root / "main.py"))}')
    run_bash_cmd(f'sudo chown -R {shlex.quote(owner)}:{shlex.quote(owner)} {parent_arg}')
    tmp.unlink()

---

## Explore the Environment

First, verify that the Notebook is connected to the *System Interpreter* of the container image.

In [ ]:
assert sys.executable == "/usr/bin/python3", (
    "This notebook expects the 'Python 3 — Ubuntu System' kernel "
    f"(/usr/bin/python3), but it is running on {sys.executable}."
)

Following commands explore the machine environment and confirm that the notebook is running on the shared system interpreter.


In [ ]:
run_bash_cmd(". /etc/os-release && printf '%s %s\n' \"$NAME\" \"$VERSION_ID\"", title='Operating system')
run_bash_cmd(f"{shlex.quote(sys.executable)} -c 'import sys; print(sys.executable); print(sys.version)'", title='Notebook interpreter')
run_bash_cmd("whoami", title='Current container user')
run_bash_cmd("id", title='Current user identity and groups')
run_bash_cmd("printf 'HOME=%s\nPWD=%s\n' \"$HOME\" \"$PWD\"", title='Current shell paths')
run_bash_cmd("python3 -m pip list", title='Installed packages from the system interpreter')

Following commands inspect the registered Linux users and compare what Bob and Alice see for identity, home directory, and default interpreter.


In [ ]:
run_bash_cmd("getent passwd bob alice", title='Registered demo users')

for who in ("bob", "alice"):
    run_bash_cmd_as_user(who, "whoami && id && echo HOME=$HOME", title=f"As {who}")

run_bash_cmd_as_user('bob', "python3 -c 'import sys; print(sys.executable)'", title="Bob's default python3")
run_bash_cmd_as_user('alice', "python3 -c 'import sys; print(sys.executable)'", title="Alice's default python3")

This code block focuses on permissions and shows group membership plus whether each user can use `sudo` non-interactively.


In [ ]:
run_bash_cmd("for user in bob alice; do groups \"$user\"; done", title='Group membership')

# Bob can run sudo non-interactively; Alice cannot.
run_bash_cmd_as_user("bob", "sudo -n id 2>&1 || echo 'sudo refused for bob'", title="bob: sudo -n id")
run_bash_cmd_as_user("alice", "sudo -n id 2>&1 || echo 'sudo refused for alice'", title="alice: sudo -n id")

> **Linux user identity and Python environment are separate concepts.**
> Switching Linux users does not, on its own, change which Python
> interpreter or which `site-packages` you are pointed at. That is exactly
> what the rest of the notebook explores.


---

## Demo Projects and Conflicting Dependencies

Now, we are creating the three targets projects including **Bob's legacy server**, **Alice FastAPI v1** and **Alice FastAPI v2** on the shared machine. 

These projects create two kinds of **dependency conflicts** (⚠️):

1) **`requests==2.0.0`** (Bob's legacy server) vs. **`requests==2.32.3`** (Alice FastAPI v1) 
2) **`fastapi==0.68.x`** (Alice FastAPI v1) vs. **`fastapi==0.111.x`** (Alice FastAPI v2).



### Bob's legacy server

The first code block defines Bob's app as an embedded Python string. The second writes that string into Bob's project directory as `main.py`.


In [ ]:
BOB_LEGACY = Path('/home/bob/projects/legacy-server')

bob_code = '''# Bob's legacy server.
import sys
import requests

print('Bob Legacy Server')
print(f'  Python  : {sys.executable}')
print(f'  requests: {requests.__version__}')

# Old-style requests API check; no network required.
req = requests.Request('GET', 'https://example.invalid/')
print(f'  built a  {req.method} request for {req.url}')
'''


With Bob's legacy server code defined, this next code block writes the project files to the shared machine.

In [ ]:
write_project(BOB_LEGACY, owner='bob', code=bob_code)
run_bash_cmd('ls -la /home/bob/projects/legacy-server', title="Bob's project")


### Alice FastAPI v1

The first code block defines Alice's older FastAPI app as an embedded Python string. The second writes it to her `fastapi-v1` project directory. It will later need FastAPI `0.68.2` and Pydantic 1.


In [ ]:
ALICE_V1 = Path('/home/alice/projects/fastapi-v1')

alice_v1_code = '''# Alice FastAPI project 1 (older FastAPI, Pydantic 1).
import sys
import fastapi
import requests
from fastapi import FastAPI

app = FastAPI(title='alice-fastapi-v1')


@app.get('/')
def root():
    return {
        'project':  'alice-fastapi-v1',
        'fastapi':  fastapi.__version__,
        'requests': requests.__version__,
    }


if __name__ == '__main__':
    print('Alice FastAPI v1')
    print(f'  Python  : {sys.executable}')
    print(f'  fastapi : {fastapi.__version__}')
    print(f'  requests: {requests.__version__}')
    routes = [r.path for r in app.routes if hasattr(r, 'path')]
    print(f'  routes  : {routes}')
'''


With Alice FastAPI v1 defined, this next code block writes the project files to the shared machine.


In [ ]:
write_project(ALICE_V1, owner='alice', code=alice_v1_code)
run_bash_cmd('ls -la /home/alice/projects/fastapi-v1', title='Alice v1 project')


### Alice FastAPI v2

The first code block defines Alice's newer FastAPI app as an embedded Python string. The second writes it to her `fastapi-v2` project directory. It will later need FastAPI `0.111.1` and Pydantic 2.


In [ ]:
ALICE_V2 = Path('/home/alice/projects/fastapi-v2')

alice_v2_code = '''# Alice FastAPI project 2 (newer FastAPI, Pydantic 2).
import sys
import fastapi
import requests
from fastapi import FastAPI

app = FastAPI(title='alice-fastapi-v2')


@app.get('/')
def root():
    return {
        'project':  'alice-fastapi-v2',
        'fastapi':  fastapi.__version__,
        'requests': requests.__version__,
    }


if __name__ == '__main__':
    print('Alice FastAPI v2')
    print(f'  Python  : {sys.executable}')
    print(f'  fastapi : {fastapi.__version__}')
    print(f'  requests: {requests.__version__}')
    routes = [r.path for r in app.routes if hasattr(r, 'path')]
    print(f'  routes  : {routes}')
'''


With Alice FastAPI v2 defined, this next code block writes the project files to the shared machine.


In [ ]:
write_project(ALICE_V2, owner='alice', code=alice_v2_code)
run_bash_cmd('ls -la /home/alice/projects/fastapi-v2', title='Alice v2 project')


In [ ]:
# Show the source of each app so the reader never needs to leave the notebook.
for label, path in [
    ('Bob — legacy-server/main.py', BOB_LEGACY / 'main.py'),
    ('Alice — fastapi-v1/main.py',  ALICE_V1   / 'main.py'),
    ('Alice — fastapi-v2/main.py',  ALICE_V2   / 'main.py'),
]:
    run_bash_cmd(f'cat {shlex.quote(str(path))}', title=label)


## Section 4 — Bob's app on the system interpreter

Bob's application was installed system-wide when we built the image
(`pip install requests==2.0.0` into the system `site-packages`).
Running it against `/usr/bin/python3` should Just Work.


In [ ]:
run_bash_cmd(
    "/usr/bin/python3 -c \"import requests, os; print('version :', requests.__version__); print('location:', os.path.dirname(requests.__file__))\"",
    title='System requests location',
)

run_bash_cmd('/usr/bin/python3 /home/bob/projects/legacy-server/main.py', title="Bob's app on /usr/bin/python3")


> The system interpreter's `site-packages` (under `/usr/lib/...` or
> `/usr/local/lib/...`) is **shared state**. Every user and every project
> that runs `/usr/bin/python3` sees exactly the packages installed there,
> at exactly the versions installed there.


## Section 5 — Alice needs a newer `requests`, system-wide

Alice's FastAPI projects were originally supposed to require
`requests==2.34.2`, but that release does not exist on PyPI. We use the
closest real release, **`2.32.3`**, and print the actual version to keep
the substitution visible.

Alice cannot install packages system-wide herself (no sudo). So imagine
that she asks Bob to do it for her. Bob obliges, without thinking about
what else uses `requests`.


In [ ]:
# Bob upgrades requests system-wide for Alice.
run_bash_cmd(
    'sudo /usr/bin/python3 -m pip install --upgrade requests==2.32.3',
    title='Bob upgrades system requests to 2.32.3',
)

run_bash_cmd(
    "/usr/bin/python3 -c \"import requests; print('system requests now at', requests.__version__)\"",
    title='Verify the new system version',
)

run_bash_cmd('/usr/bin/python3 /home/bob/projects/legacy-server/main.py', title="Bob's app after Alice's upgrade")


> Bob's app still runs, but the host-side check above shows that the shared interpreter now exposes the wrong `requests` version for Bob's project.


> **A system Python environment is shared state.** Changing a dependency
> for one user changes it for every user and every project on the machine.


## Section 6 — Restore Bob's environment

The teaching notebook must be re-runnable, so we roll the system
interpreter back to `requests==2.0.0`. In real life this is exactly the
kind of firefighting that motivates isolated environments.


In [ ]:
run_bash_cmd(
    'sudo /usr/bin/python3 -m pip install --force-reinstall requests==2.0.0',
    title='Restore requests==2.0.0',
)

run_bash_cmd('/usr/bin/python3 /home/bob/projects/legacy-server/main.py', title="Bob's app after restore")


## Section 7 — Alice uses `pip install --user`

The next fix: Alice stops touching the system interpreter's
`site-packages` and installs into her personal user site directory
(`~/.local/lib/pythonX.Y/site-packages`). Bob's environment is now
untouchable by Alice — but each Linux user still has only **one** package
scope for **all** their projects.


In [ ]:
# Alice installs requests==2.32.3 into her own user site directory.
run_bash_cmd_as_user(
    'alice',
    'python3 -m pip install --user --upgrade requests==2.32.3',
    title='alice: pip install --user requests==2.32.3',
)

run_bash_cmd_as_user(
    'alice',
    "python3 -c 'import site, sys; print(\"user site:\", site.getusersitepackages()); print(\"user base:\", site.getuserbase()); print(\"exe      :\", sys.executable)'",
    title='alice: interpreter + user site',
)


In [ ]:
# Alice's view of requests.
run_bash_cmd_as_user(
    'alice',
    "python3 -c 'import requests, os; print(\"version :\", requests.__version__); print(\"location:\", os.path.dirname(requests.__file__))'",
    title='alice: which requests wins?',
)

# Bob's view of requests — untouched by Alice's --user install.
run_bash_cmd(
    "/usr/bin/python3 -c \"import requests, os; print('version :', requests.__version__); print('location:', os.path.dirname(requests.__file__))\"",
    title='bob (system): which requests wins?',
)


In [ ]:
# Bob's legacy app is safe.
run_bash_cmd('/usr/bin/python3 /home/bob/projects/legacy-server/main.py', title="Bob's legacy app is unaffected")


> `pip install --user` isolates users **from each other**. Bob and
> Alice can now hold different versions of `requests` on the same machine.


### But `--user` does not isolate projects from each other

Alice has two FastAPI projects. They need incompatible FastAPI versions:

```
alice-fastapi-v1
    ├── fastapi 0.68.2  (Pydantic 1)
    └── requests 2.32.x

alice-fastapi-v2
    ├── fastapi 0.111.1 (Pydantic 2)
    └── requests 2.32.x
```

Both projects use the same `/usr/bin/python3` and the same
`~/.local/lib/pythonX.Y/site-packages`. Whichever version Alice installed
last is the version *both* projects see.


In [ ]:
# Install FastAPI 0.68.2 for project v1.
run_bash_cmd_as_user(
    'alice',
    "python3 -m pip install --user 'fastapi==0.68.2' 'pydantic<2'",
    title='alice: install fastapi 0.68.2 (--user)',
)

# Run project v1 — should work.
run_bash_cmd_as_user(
    'alice',
    'python3 /home/alice/projects/fastapi-v1/main.py',
    title='alice: run fastapi-v1 with fastapi 0.68.2',
)

# Attempt to run project v2 with the same environment — v2 expects 0.111.x.
run_bash_cmd_as_user(
    'alice',
    'python3 /home/alice/projects/fastapi-v2/main.py',
    title='alice: run fastapi-v2 while fastapi 0.68.2 is installed',
)


In [ ]:
# Now Alice upgrades to FastAPI 0.111.1 for project v2.
run_bash_cmd_as_user(
    'alice',
    "python3 -m pip install --user --upgrade 'fastapi==0.111.1' 'pydantic>=2'",
    title='alice: upgrade to fastapi 0.111.1 (--user)',
)

# v2 works now.
run_bash_cmd_as_user(
    'alice',
    'python3 /home/alice/projects/fastapi-v2/main.py',
    title='alice: run fastapi-v2 with fastapi 0.111.1',
)

# But v1 no longer sees the version it was written against.
run_bash_cmd_as_user(
    'alice',
    'python3 /home/alice/projects/fastapi-v1/main.py',
    title='alice: run fastapi-v1 AFTER upgrading FastAPI globally',
)


> `--user` isolates users, **but not projects**. Alice cannot keep two
> incompatible FastAPI versions in her single `~/.local` site directory.
> This is the wall that virtual environments were designed to knock down.


## Section 8 — Per-project virtual environments

Each project gets its own `python3 -m venv .venv`. Each `.venv` has its
own `site-packages`, its own installed dependency versions, and its own
`bin/python`. Nothing else on the machine is touched.

We create three virtual environments, each owned by the project owner:

- `/home/bob/projects/legacy-server/.venv`
- `/home/alice/projects/fastapi-v1/.venv`
- `/home/alice/projects/fastapi-v2/.venv`


In [ ]:
# Bob: legacy server venv with the ancient requests.
run_bash_cmd_as_user(
    'bob',
    'python3 -m venv /home/bob/projects/legacy-server/.venv',
    title='bob: create legacy .venv',
)
run_bash_cmd_as_user(
    'bob',
    "/home/bob/projects/legacy-server/.venv/bin/pip install --quiet --upgrade pip && /home/bob/projects/legacy-server/.venv/bin/pip install --quiet 'requests==2.0.0'",
    title='bob: install requests==2.0.0 into legacy .venv',
)

# Alice v1: older FastAPI on Pydantic 1.
run_bash_cmd_as_user(
    'alice',
    'python3 -m venv /home/alice/projects/fastapi-v1/.venv',
    title='alice: create fastapi-v1 .venv',
)
run_bash_cmd_as_user(
    'alice',
    "/home/alice/projects/fastapi-v1/.venv/bin/pip install --quiet --upgrade pip && /home/alice/projects/fastapi-v1/.venv/bin/pip install --quiet 'fastapi==0.68.2' 'pydantic<2' 'requests==2.32.3'",
    title='alice: install v1 dependencies',
)

# Alice v2: newer FastAPI on Pydantic 2.
run_bash_cmd_as_user(
    'alice',
    'python3 -m venv /home/alice/projects/fastapi-v2/.venv',
    title='alice: create fastapi-v2 .venv',
)
run_bash_cmd_as_user(
    'alice',
    "/home/alice/projects/fastapi-v2/.venv/bin/pip install --quiet --upgrade pip && /home/alice/projects/fastapi-v2/.venv/bin/pip install --quiet 'fastapi==0.111.1' 'requests==2.32.3'",
    title='alice: install v2 dependencies',
)


### The final state — three projects, three interpreters

Each application now runs from its own project's `.venv/bin/python`. The
interpreter, the FastAPI version, and the `requests` version are all
independent.


In [ ]:
apps = [
    ('Bob Legacy',
     'bob',
     '/home/bob/projects/legacy-server/.venv/bin/python',
     '/home/bob/projects/legacy-server/main.py'),
    ('Alice FastAPI V1',
     'alice',
     '/home/alice/projects/fastapi-v1/.venv/bin/python',
     '/home/alice/projects/fastapi-v1/main.py'),
    ('Alice FastAPI V2',
     'alice',
     '/home/alice/projects/fastapi-v2/.venv/bin/python',
     '/home/alice/projects/fastapi-v2/main.py'),
]

for label, user, interp, script in apps:
    run_bash_cmd_as_user(user, f'{interp} {script}', title=label)


## Section 9 — Interactive user switch

A small ipywidgets dropdown lets you run a predefined safe command as
either Bob or Alice. Only whitelisted commands are exposed — the
notebook does not offer a general shell interface.

> Switching Linux users **does not** automatically switch Python
> environments. Both Bob and Alice, by default, use `/usr/bin/python3`
> and its system `site-packages`. The virtual environments above are
> what changed that.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

SAFE_COMMANDS = {
    'whoami':                     'whoami',
    'id':                         'id',
    'pwd (home)':                 'pwd',
    'python --version':           'python3 --version',
    'requests version (system)':  "python3 -c 'import requests; print(requests.__version__)'",
    'which python':               'command -v python3',
}

user_dd = widgets.Dropdown(options=['bob', 'alice'], value='bob',  description='User:')
cmd_dd  = widgets.Dropdown(options=list(SAFE_COMMANDS), value='whoami', description='Command:')
button  = widgets.Button(description='Run', button_style='primary')
out     = widgets.Output()


def on_click(_button):
    out.clear_output()
    with out:
        cmd = SAFE_COMMANDS[cmd_dd.value]
        run_bash_cmd_as_user(user_dd.value, cmd, title=f'{user_dd.value}: {cmd}')


button.on_click(on_click)
display(widgets.VBox([widgets.HBox([user_dd, cmd_dd, button]), out]))


## Section 10 — Dependency scope visualisation

**Initial state — everything on the system interpreter**

| User  | Project        | Environment | Dependency        |
| ----- | -------------- | ----------- | ----------------- |
| Bob   | Legacy         | System      | `requests 2.0.0`  |
| Alice | FastAPI V1     | System      | `requests 2.34.x` *(wants)* |
| Alice | FastAPI V2     | System      | `fastapi X or Y`  |

**After `pip install --user` for Alice**

| User  | Project        | Environment          | Dependency               |
| ----- | -------------- | -------------------- | ------------------------ |
| Bob   | Legacy         | System               | `requests 2.0.0`         |
| Alice | FastAPI V1     | `~/.local/...`       | `fastapi 0.68.2`         |
| Alice | FastAPI V2     | `~/.local/...`       | `fastapi 0.111.1` — **overwrites V1's copy** |

**After per-project virtual environments**

| Project        | Python environment                                    | Dependencies                        |
| -------------- | ----------------------------------------------------- | ----------------------------------- |
| Bob Legacy     | `/home/bob/projects/legacy-server/.venv`              | `requests==2.0.0`                   |
| Alice FastAPI V1 | `/home/alice/projects/fastapi-v1/.venv`             | `fastapi==0.68.2`, `pydantic<2`, `requests==2.32.3` |
| Alice FastAPI V2 | `/home/alice/projects/fastapi-v2/.venv`             | `fastapi==0.111.1`, `pydantic>=2`, `requests==2.32.3` |
